In [ ]:
%%writefile dna.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 1000

__global__ void dna_count(char* arr, int* result) {
    int gid = threadIdx.x + blockIdx.x * blockDim.x;

    if (gid < N) {
        char c = arr[gid];

        int index =
            (c == 'C') * 1 +
            (c == 'G') * 2 +
            (c == 'T') * 3;

        atomicAdd(&result[index], 1);
    }
}

int main() {
    srand(time(NULL));

    char* h_arr; cudaMallocHost((void**)&h_arr, N * sizeof(char));

    for (int i = 0; i < N; i++) {
        int num = rand() % 4;

        if (num == 0) {
            h_arr[i] = 'A';
        } else if (num == 1) {
            h_arr[i] = 'C';
        } else if (num == 2) {
            h_arr[i] = 'G';
        } else {
            h_arr[i] = 'T';
        }
    }


    char* d_arr; cudaMalloc((void**)&d_arr, N * sizeof(char));
    cudaMemcpy(d_arr, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);

    int* h_result; cudaMallocHost((void**)&h_result, 4 * sizeof(int));
    h_result[0] = 0; h_result[1] = 0; h_result[2] = 0; h_result[3] = 0;

    int* d_result; cudaMalloc((void**)&d_result, 4 * sizeof(int));
    cudaMemcpy(d_result, h_result, 4 * sizeof(int), cudaMemcpyHostToDevice);

    int threadNum = 64;
    int blockNum = (N + threadNum - 1) / threadNum;
    dna_count<<<blockNum, threadNum>>>(d_arr, d_result);

    cudaDeviceSynchronize();

    cudaMemcpy(h_result, d_result, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("A: %d, C: %d, G: %d, T: %d", h_result[0], h_result[1], h_result[2], h_result[3]);

    cudaFreeHost(h_arr);
    cudaFreeHost(h_result);

    cudaFree(d_arr);
    cudaFree(d_result);

}


Overwriting dna.cu


In [ ]:
!nvcc dna.cu -o Dna

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./Dna

A: 268, C: 257, G: 233, T: 242

In [ ]:
%%writefile rna.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 1000

__global__ void rna_to_dna(char* rna) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;

    if (gid < N) {
        if (rna[gid] == 'U') {
            rna[gid] = 'T';
        }
    }
}

int main() {
    srand(time(NULL));

    char* h_arr; cudaMallocHost((void**)&h_arr, N * sizeof(char));

    for (int i = 0; i < N; i++) {
        int num = rand() % 4;

        if (num == 0) {
            h_arr[i] = 'A';
        } else if (num == 1) {
            h_arr[i] = 'C';
        } else if (num == 2) {
            h_arr[i] = 'G';
        } else {
            h_arr[i] = 'U';
        }
    }

    printf("The RNA: \n");
    for (int i = 0; i < N; i++) {
        printf("%c", h_arr[i]);
    }
    printf("\n");

    char* d_arr; cudaMalloc((void**)&d_arr, N * sizeof(char));
    cudaMemcpy(d_arr, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);

    int threadNum = 32;
    int blockNum = (N + threadNum - 1) / threadNum;
    rna_to_dna<<<blockNum, threadNum>>>(d_arr);

    cudaDeviceSynchronize();

    cudaMemcpy(h_arr, d_arr, N * sizeof(char), cudaMemcpyDeviceToHost);

    printf("The DNA: \n");
    for (int i = 0; i < N; i++) {
        printf("%c", h_arr[i]);
    }
    printf("\n");

    cudaFreeHost(h_arr);
    cudaFree(d_arr);
}



Overwriting rna.cu


In [ ]:
!nvcc rna.cu -o Rna

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./Rna

The RNA: 
GCUCAUAUUCUUAGCGCGGUAAAUACACGAAACUCGUCCGGAGGGUCUCUUCAUCAACGGCGGUGUCCAGUUUCCCCGCGGAAGUCGUGACAUUUCGAGGGCCCUUUACAGUAGCUUAGGAUGUGCAACGGUAACUUAUAACAAACUUCCCGAUCGAGGCAACAAGUUGGAGAAGACCUUGCCGAGACAGGCGUCUCCGUUGCUGUUAAUUGAAAAGACUGAACUCACGGAGCGCUCCUGAGAAGCACCGAAGACCCCGAUUGACUAGAUAAGCAAGCCUUCUCCAGUCCUCACCCACUCCACUCGUUUCGGGGUAGCUAGGCGUGUUAUACUGAAAUUUACCUUAUGCGGAAUGUCGGCCUGACGACGUAGACCACAGGGAGGUCCAUAGAUAAAUACCUCUUGCUUCCCACAGACACAAUAUUAUGAAUUCUGUAGGCUUCCUACAAGAACAUCAGUAGUUUGGGGAAUUUCAUCGUCAACGAAUCGGCACCUUUCCUCCGAGUUUCUACUCUUCGAUCGUGUGGGUUCAAUAGGACUUCCGGACUUGGAACGUUACGAGGACUUAUAUGGAAACCAAUGAUAGGUUUGUCAUUUGUGUGACUACAGCACAACAGUUCUCAAGAUCGGACCAGACAAUCAACACCGCCAAGAAAAAGUGGAUGUUUUUGAUUGUAUGCAGCUGGUUGCGAUGAGCUCAUACGUUGUGACUGGGAACUGUCUGAUACGGGCGUACGGAAGCUAACAAUAUGCGACCCUUUCCCACGGUGGCAGCAUCAGCUAUCCAGGUGCAUGAAAGUGCAGUCGGGUCAGCUAGUGAUACUUUAUUGGGUGAGUUCGGGGAUCAGAUGUUAUUUUGUCCCCUCUUAACGGUUGAUAAGGAGGUCGGUCUAGAAUAUAAAGUUCCUCCUUUUCGGGAAGCAGUAGGACGCCCAAGCUUGUUCGAUCUUCCACUUGGGGUAUCCUCAACUGAGAGUUUGUCUUGGUAAC

In [13]:
%%writefile prtm.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>


__global__ void prtm_counter(char* prtm, float* count, float* weights, int N) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;

    if (gid < N) {
        float val = weights[prtm[gid]];

        atomicAdd(count, val);
    }
}

int prtm(char* h_arr) {
    float h_weights[128];

    h_weights['A'] = 71.03711;
    h_weights['C'] = 103.00919;
    h_weights['D'] = 115.02694;
    h_weights['E'] = 129.04259;
    h_weights['F'] = 147.06841;
    h_weights['G'] = 57.02146;
    h_weights['H'] = 137.05891;
    h_weights['I'] = 113.08406;
    h_weights['K'] = 128.09496;
    h_weights['L'] = 113.08406;
    h_weights['M'] = 131.04049;
    h_weights['N'] = 114.04293;
    h_weights['P'] = 97.05276;
    h_weights['Q'] = 128.05858;
    h_weights['R'] = 156.10111;
    h_weights['S'] = 87.03203;
    h_weights['T'] = 101.04768;
    h_weights['V'] = 99.06841;
    h_weights['W'] = 186.07931;
    h_weights['Y'] = 163.06333;

    float* d_weights; cudaMalloc((void**)&d_weights, 128 * sizeof(float));
    cudaMemcpy(d_weights, h_weights, 128 * sizeof(float), cudaMemcpyHostToDevice);

    int N = strlen(h_arr);

    char* d_arr; cudaMalloc((void**)&d_arr, N * sizeof(char));
    cudaMemcpy(d_arr, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);

    float* d_count; cudaMalloc((void**)&d_count, sizeof(float));
    cudaMemset(d_count, 0, sizeof(float));

    int threadNum = 32;
    int blockNum = (N + threadNum - 1) / threadNum;
    prtm_counter<<<blockNum, threadNum>>>(d_arr, d_count, d_weights, N);

    cudaDeviceSynchronize();

    float h_count;
    cudaMemcpy(&h_count, d_count, sizeof(float), cudaMemcpyDeviceToHost);

    printf("The sum is %f \n", h_count);

    cudaFree(d_arr);
    cudaFree(d_count);

    return h_count;
}

int main() {
    char str[] = "SKADYEK";
    prtm(str);
}



Overwriting prtm.cu


In [14]:
!nvcc prtm.cu -o Prtm

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [15]:
!./Prtm

The sum is 821.391968 


In [ ]:
%%writefile hamm.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 1000

__global__ void hamm(char* d_arr_1, char* d_arr_2, int* count) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;

    if (gid < N) {
        if (d_arr_1[gid] != d_arr_2[gid]) {
            atomicAdd(count, 1);
        }
    }
}

int main() {
    srand(time(NULL));

    char* h_arr_1; cudaMallocHost((void**)&h_arr_1, N * sizeof(char));
    char* h_arr_2; cudaMallocHost((void**)&h_arr_2, N * sizeof(char));

    for (int i = 0; i < N; i++) {
        int num_1 = rand() % 4;

        if (num_1 == 0) {
            h_arr_1[i] = 'A';
        } else if (num_1 == 1) {
            h_arr_1[i] = 'C';
        } else if (num_1 == 2) {
            h_arr_1[i] = 'G';
        } else {
            h_arr_1[i] = 'U';
        }

        int num_2 = rand() % 4;

        if (num_2 == 0) {
            h_arr_2[i] = 'A';
        } else if (num_2 == 1) {
            h_arr_2[i] = 'C';
        } else if (num_2 == 2) {
            h_arr_2[i] = 'G';
        } else {
            h_arr_2[i] = 'U';
        }
    }

    char* d_arr_1; cudaMalloc((void**)&d_arr_1, N * sizeof(char));
    cudaMemcpy(d_arr_1, h_arr_1, N * sizeof(char), cudaMemcpyHostToDevice);

    char* d_arr_2; cudaMalloc((void**)&d_arr_2, N * sizeof(char));
    cudaMemcpy(d_arr_2, h_arr_2, N * sizeof(char), cudaMemcpyHostToDevice);

    int* d_count; cudaMalloc((void**)&d_count, sizeof(int));
    cudaMemset(d_count, 0, sizeof(int));

    int threadNum = 32;
    int blockNum = (N + threadNum - 1) / threadNum;
    hamm<<<blockNum, threadNum>>>(d_arr_1, d_arr_2, d_count);

    cudaDeviceSynchronize();

    int h_count;
    cudaMemcpy(&h_count, d_count, sizeof(int), cudaMemcpyDeviceToHost);

    printf("The number of differences is %d \n", h_count);

    cudaFreeHost(h_arr_1);
    cudaFreeHost(h_arr_2);

    cudaFree(d_arr_1);
    cudaFree(d_arr_2);
    cudaFree(d_count);

    return h_count;
}



Overwriting hamm.cu


In [ ]:
!nvcc hamm.cu -o Hamm

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./Hamm

The number of differences is 756 
